# 09.3 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/03-embeddings/exercises.ipynb)

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import os, logging, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, str(data_path("download.py").parent))

## Exercise 1: by hand

Cosine similarity of $(1, 2, 2)$ and $(2, 1, 2)$. Then normalise both to length 1 and check that the dot product matches.

In [ ]:
# your code here

## Setup for exercises 2–4

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")   # ~90 MB download, once
print(f"device: {encoder.device}, {sum(p.numel() for p in encoder.parameters()):,} parameters, "
      f"{encoder.get_sentence_embedding_dimension()} numbers per text")

def embed(texts):
    return encoder.encode(list(texts), normalize_embeddings=True, batch_size=64, show_progress_bar=False)

In [ ]:
from download import load_retail

raw = load_retail()
names = raw["Description"].dropna().str.strip()
counts = names[names.str.len() > 3].value_counts()
not_products = ["POSTAGE", "MANUAL", "DOTCOM POSTAGE", "CARRIAGE", "BANK CHARGES",
                "DISCOUNT", "AMAZON FEE", "SAMPLES", "CRUK COMMISSION"]
# product names are written in capitals; lower-case entries are stock notes ("found", "damaged")
mostly_caps = counts.index.str.count(r"[A-Z]") >= counts.index.str.count(r"[a-z]")
products = counts[(counts >= 5) & mostly_caps & ~counts.index.isin(not_products)].index.to_series(name="name")
products = products.reset_index(drop=True)
print(f"{len(products):,} ShopWave products")
products.sample(8, random_state=1).tolist()

In [ ]:
E = embed(products.str.lower())

def search(query, k=5):
    scores = E @ embed([query])[0]
    best = scores.argsort()[::-1][:k]
    return [(products.iloc[i], round(float(scores[i]), 3)) for i in best]

## Exercise 2: your own queries

Try five queries. Compare with keyword search (`products.str.lower().str.contains(query)`).

In [ ]:
# your code here

## Exercise 3: near-duplicates

Find pairs of products with cosine similarity above 0.95. Print ten. Same product twice, or different products?

In [ ]:
# your code here

## Exercise 4: search reviews

Find the 3 test reviews most similar to *"it arrived broken and I want my money back"*.

In [ ]:
from download import load_reviews
from sklearn.model_selection import train_test_split

reviews = load_reviews()          # first run downloads one 117 MB file, then cached
train, test = train_test_split(reviews, test_size=0.2, random_state=42, stratify=reviews["label"])
print(f"{len(reviews):,} reviews ({reviews['label'].mean():.0%} positive): "
      f"train {len(train):,} / test {len(test):,}")
reviews[["text", "label"]].head()

In [ ]:
# your code here